# Extract structured fields from a vendor invoice with Document Intelligence and summarize the result with GPT-4o-mini

Your ops team gets 50 vendor invoices a day as PDFs. The team currently pays an offshore contractor to retype line items into a spreadsheet, the spreadsheet is the source of truth for monthly vendor reconciliation, and the contractor occasionally fat-fingers a quantity. You have one session to wire Document Intelligence's prebuilt-invoice model against a 5-page sample, extract vendor and totals and line items as structured JSON, run a sanity check that the line-item amounts sum to the invoice total, and produce a GPT-4o-mini summary that the ops lead can paste into a Slack channel for vendor approvals. The deliverable is the JSON plus the one-sentence summary.

What you will build:

- A Foundry hub plus project in `eastus`, plus a `gpt-4o-mini` Standard deployment for the post-extraction summary.
- An Azure AI Document Intelligence resource (`kind=FormRecognizer` on the management plane, S0 tier) for the prebuilt-invoice analyzer.
- A 5-page sample invoice analyzed through `prebuilt-invoice`, with VendorName, InvoiceTotal, and Items pulled into Python state.
- A sum-of-line-items vs invoice-total sanity check within $0.01 tolerance, the canonical post-extraction data-quality control.
- A GPT-4o-mini summary of the structured JSON in the shape `<vendor> invoice for <total> across <N> line items.`.

**Time.** Plan for about 55 minutes hands-on. Foundry provisioning runs 6 to 9 minutes; the Document Intelligence resource itself is fast (~1 minute); the analyzer call is the slow part (20 to 40 seconds for a 5-page PDF). Budget up to 90 minutes total.

**Cost.** Document Intelligence prebuilt-invoice is $1.50 per 1,000 pages, so a 5-page analyze call is $0.0075. Two debugging runs are roughly $0.015. The GPT-4o-mini summary at the end is cents (~$0.0001 per call). Foundry orchestration, RBAC, and ARM are free. A clean session lands around $0.02; even heavy debugging tops out near $0.30. The trap is debugging the field walk: prebuilt-invoice returns a deeply nested object shape with currency types inside arrays inside fields, and you may iterate the analyzer call three times before you find the right access path.

This lab maps to AI-103 Domain 5: Implement information extraction solutions (13% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs for Document Intelligence and the
# AOAI summary call.
# - azure-ai-formrecognizer was renamed to azure-ai-documentintelligence in
#   2023. The lab installs the new package; do NOT install the legacy one.
# - azure-ai-projects 2.0 (March 2026) absorbed azure-ai-agents.

!pip install --quiet labrun-checks==0.6.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 azure-ai-documentintelligence>=1.0.0 openai>=2.0.0 requests>=2.32.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.

import atexit
import getpass
import json
import time

import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.ai.documentintelligence import DocumentIntelligenceClient
from openai import AzureOpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "document-intelligence-extraction"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
DOC_INTEL_NAME = f"labrun-{LAB_ID}-docintel"
MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30

# Resolved during setup.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
DOC_INTEL_ENDPOINT = None
AZURE_CREDENTIAL = None

# State populated by tasks for downstream checkpoints.
ANALYSIS_RESULT = None
VENDOR_NAME = None
INVOICE_TOTAL = None
LINE_ITEMS = []
LINE_ITEMS_SUM = None
GPT_SUMMARY = None

# Sample invoice hosted in the public labrun-labs assets repo at a pinned path.
SAMPLE_INVOICE_URL = (
    "https://raw.githubusercontent.com/labrun-labs/assets/main/"
    "azure-ai-103/lab-10/sample-invoice-5pg.pdf"
)
LOCAL_PDF_PATH = "/tmp/labrun-doc-intel-invoice.pdf"

# Pricing for wallet checks.
DOC_INTEL_PER_1K_PAGES_USD = 1.50
CHAT_PRICE_PER_1M_INPUT_USD = 0.15
CHAT_PRICE_PER_1M_OUTPUT_USD = 0.60

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. This lab pins {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# No critical resources: Document Intelligence bills per page only, and the
# GPT-4o-mini Standard deployment does not bill at zero traffic. Cleanup tears
# every resource down for orphan-scan hygiene. Reverse-creation order:
# Document Intelligence resource, deployment, project, hub, resource group.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="document_intelligence_resource",
        id=DOC_INTEL_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account delete "
            f"--resource-group {RESOURCE_GROUP} --name {DOC_INTEL_NAME}"
        ),
    ),
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up Foundry, deploy GPT-4o-mini, and provision a Document Intelligence resource

The Foundry stack is the same shape as Lab 01. The Document Intelligence resource is a Cognitive Services account with `kind=FormRecognizer`. Microsoft renamed the product to "Document Intelligence" in 2023 but preserved the legacy `kind` value on the management plane for backward compatibility; the validator asserts that exact string.

Build it in this order:

1. Create the resource group, hub, and project with the lab tag.
2. Discover the attached Azure OpenAI account from the hub's connections. Deploy `gpt-4o-mini` at Standard SKU 30k TPM under the attached AOAI account.
3. Provision the Document Intelligence resource: `kind=FormRecognizer`, `sku=S0`, `location=eastus`, lab tag. Capture `DOC_INTEL_ENDPOINT` from the resource's `properties.endpoint`.

**SDK migration callout.** The Python package `azure-ai-formrecognizer` is deprecated. The lab pins `azure-ai-documentintelligence>=1.0.0` and imports `DocumentIntelligenceClient`. Tutorials and old Microsoft Learn modules still reference `FormRecognizerClient`; that is the deprecated surface and will not be installed by the pip cell above. The management-plane `kind` string (`FormRecognizer`) is unrelated to the SDK package rename; it stays as is.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision the Foundry stack and the Document Intelligence resource.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()

# Discover the attached Azure OpenAI account.
for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
if not AOAI_ACCOUNT_NAME:
    print("Could not find an attached Azure OpenAI account on the hub.")
    raise SystemExit(1)

aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

deployment_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": MODEL_NAME, "version": MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the model deployment go through Succeeded purgatory, about 1 to 2 minutes...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, deployment_payload,
).result()
print(f"GPT-4o-mini deployment ready: {DEPLOYMENT_NAME}")

# Document Intelligence resource. kind=FormRecognizer is the legacy string
# preserved by Microsoft on the management plane even after the rename.
doc_intel_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "FormRecognizer",
    "sku": {"name": "S0"},
    "identity": {"type": "SystemAssigned"},
    "properties": {"public_network_access": "Enabled"},
}
print("Provisioning the Document Intelligence resource, about 1 minute...")
# YOUR CODE: Provision the Document Intelligence resource via
# cs_client.accounts.begin_create_or_update(RESOURCE_GROUP, DOC_INTEL_NAME,
# doc_intel_payload). Call .result() on the poller. Store the result in
# doc_intel.

DOC_INTEL_ENDPOINT = doc_intel.properties.endpoint
print(f"Document Intelligence resource ready: {DOC_INTEL_NAME}")
print(f"Endpoint: {DOC_INTEL_ENDPOINT}")
print(f"Kind: {doc_intel.kind} (legacy string; product was renamed in 2023)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Document Intelligence resource exists with kind=FormRecognizer,
# provisioningState=Succeeded, location=eastus, lab tag present. Then a
# data-plane get-model call confirms prebuilt-invoice is reachable.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        try:
            account = cs_client.accounts.get(RESOURCE_GROUP, DOC_INTEL_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Document Intelligence resource {DOC_INTEL_NAME!r} not found in "
                    f"{RESOURCE_GROUP!r}. Did Task 1's begin_create_or_update call run?"
                ),
            )

        if account.kind != "FormRecognizer":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Resource kind is {account.kind!r}, expected 'FormRecognizer'. "
                    f"Microsoft preserved this legacy value on the management plane after "
                    f"the product rename; do not change it."
                ),
            )

        provisioning_state = getattr(account.properties, "provisioning_state", None)
        if provisioning_state != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"provisioningState is {provisioning_state!r}, expected 'Succeeded'. "
                    f"Wait for the poller to finish (call .result() on begin_create_or_update)."
                ),
            )

        if account.location != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Resource location is {account.location!r}, expected {REGION!r}."
                ),
            )

        tags = account.tags or {}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Resource missing lab tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Without it, the orphan scan cannot find this resource on re-run."
                ),
            )

        # Data-plane reachability: confirm prebuilt-invoice answers a get-model call.
        if not DOC_INTEL_ENDPOINT:
            return CheckpointResult(
                status="fail",
                error_reason="DOC_INTEL_ENDPOINT is None. Capture it from doc_intel.properties.endpoint.",
            )
        try:
            client = DocumentIntelligenceClient(
                endpoint=DOC_INTEL_ENDPOINT, credential=AZURE_CREDENTIAL,
            )
            model = client.get_document_model("prebuilt-invoice")
        except HttpResponseError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Data-plane get_document_model('prebuilt-invoice') failed: {e.message}. "
                    f"Confirm the service principal has 'Cognitive Services User' on this resource."
                ),
            )

        model_id = getattr(model, "model_id", None)
        if model_id != "prebuilt-invoice":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"get_document_model returned model_id={model_id!r}, expected 'prebuilt-invoice'."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

If the checkpoint says "Resource not found," the create call never ran. If the kind complaint fires, you tried `kind="DocumentIntelligence"` based on the product name; Microsoft kept the legacy `kind="FormRecognizer"` on the management plane. If the get-model call fails with 401 or 403, your service principal is missing the `Cognitive Services User` role on this resource.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `cs_client.accounts.begin_create_or_update(RESOURCE_GROUP, DOC_INTEL_NAME, doc_intel_payload)`. The poller is async; call `.result()` and store in `doc_intel`. The `doc_intel_payload` variable in the cell above already has `kind="FormRecognizer"`, `sku.name="S0"`, `location=REGION`, and `tags=lab_tags`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1's `YOUR CODE` line:

```python
doc_intel = cs_client.accounts.begin_create_or_update(
    RESOURCE_GROUP, DOC_INTEL_NAME, doc_intel_payload,
).result()
```

</details>

**Wallet check.** Still at $0.00 for the Document Intelligence side. Provisioning is free; you pay only when you submit a document. The Foundry hub, project, and Standard deployment also carry no hourly fee at zero traffic. Coffee is still way ahead.

## Task 2: Analyze the 5-page invoice and pull VendorName, InvoiceTotal, and Items

Now you spend real money. The 5-page sample PDF is hosted in the public `labrun-labs/assets` GitHub repo. You download it, hand the bytes to the `prebuilt-invoice` analyzer, and wait for the LRO to finish. Typical wait for a 5-page PDF is 20 to 40 seconds.

Build it in this order:

1. Download `SAMPLE_INVOICE_URL` to `LOCAL_PDF_PATH` using `requests`.
2. Open the PDF in binary mode and read all bytes into `pdf_bytes`.
3. Submit via `client.begin_analyze_document(model_id="prebuilt-invoice", body=pdf_bytes)`. The poller returns immediately; call `.result()` to block until the analyzer finishes. Capture in `ANALYSIS_RESULT`.
4. Pull the first document: `doc = ANALYSIS_RESULT.documents[0]`. Read `doc.fields["VendorName"].content` (string), `doc.fields["InvoiceTotal"].value_currency.amount` (float), and iterate `doc.fields["Items"].value_array`.

**Field-access pattern.** Prebuilt-invoice returns `doc.fields[FIELD_NAME].value_<type>` where the type suffix matches the field's data type: `_string` for strings, `_currency` for money, `_array` for repeating fields, `_object` for nested records. Students who try `field.value` (no type suffix) get `None`. The string field also has a `.content` attribute that holds the raw extracted text; either works for `VendorName`.

In [ ]:
# NBVAL_SKIP
# Task 2: Download the sample, submit to prebuilt-invoice, parse the result.

print("Downloading the 5-page sample invoice from labrun-labs/assets...")
resp = requests.get(SAMPLE_INVOICE_URL, timeout=30)
resp.raise_for_status()
with open(LOCAL_PDF_PATH, "wb") as fh:
    fh.write(resp.content)
print(f"Saved {len(resp.content)} bytes to {LOCAL_PDF_PATH}")

with open(LOCAL_PDF_PATH, "rb") as fh:
    pdf_bytes = fh.read()

doc_intel_client = DocumentIntelligenceClient(
    endpoint=DOC_INTEL_ENDPOINT, credential=AZURE_CREDENTIAL,
)

print("Sending the 5-page invoice to Document Intelligence, takes about 30 seconds for a clean PDF...")
# YOUR CODE: Submit to prebuilt-invoice and block on the LRO.
# poller = doc_intel_client.begin_analyze_document(
#     model_id="prebuilt-invoice", body=pdf_bytes,
# )
# ANALYSIS_RESULT = poller.result()

print("Reading the prebuilt-invoice fields out of the analyzer result...")
doc = ANALYSIS_RESULT.documents[0]
print(f"First document fields: {list(doc.fields.keys())}")

vendor_field = doc.fields.get("VendorName")
VENDOR_NAME = vendor_field.content if vendor_field else None

total_field = doc.fields.get("InvoiceTotal")
if total_field and getattr(total_field, "value_currency", None):
    INVOICE_TOTAL = total_field.value_currency.amount

items_field = doc.fields.get("Items")
items_array = items_field.value_array if items_field else []

print(f"VendorName: {VENDOR_NAME!r}")
print(f"InvoiceTotal: {INVOICE_TOTAL}")
print(f"Items extracted: {len(items_array)}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: ANALYSIS_RESULT is populated, the first document has a
# non-empty VendorName, a positive InvoiceTotal, and at least 3 line items.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if ANALYSIS_RESULT is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "ANALYSIS_RESULT is None. Submit pdf_bytes via "
                    "begin_analyze_document(model_id='prebuilt-invoice', body=pdf_bytes) "
                    "and call .result() on the poller."
                ),
            )

        docs = getattr(ANALYSIS_RESULT, "documents", None) or []
        if not docs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "ANALYSIS_RESULT.documents is empty. The analyzer did not classify "
                    "the PDF as an invoice. Confirm SAMPLE_INVOICE_URL downloaded the "
                    "intended 5-page sample."
                ),
            )
        doc = docs[0]

        vendor_field = doc.fields.get("VendorName") if doc.fields else None
        if vendor_field is None:
            return CheckpointResult(
                status="fail",
                error_reason="doc.fields['VendorName'] is missing. Check the field-access pattern.",
            )
        vendor_text = vendor_field.content
        if not isinstance(vendor_text, str) or not vendor_text.strip():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "doc.fields['VendorName'].content was None or empty; check the "
                    "field-access pattern: .value_string vs .content (both should resolve "
                    "to the raw vendor string)."
                ),
            )

        total_field = doc.fields.get("InvoiceTotal") if doc.fields else None
        if total_field is None:
            return CheckpointResult(
                status="fail",
                error_reason="doc.fields['InvoiceTotal'] is missing.",
            )
        currency = getattr(total_field, "value_currency", None)
        if currency is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "doc.fields['InvoiceTotal'].value_currency is None. The currency-typed "
                    "path is .value_currency.amount, not .value."
                ),
            )
        total_amount = currency.amount
        if not isinstance(total_amount, (int, float)) or total_amount <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"InvoiceTotal.value_currency.amount is {total_amount!r}, expected a "
                    f"positive float."
                ),
            )

        items_field = doc.fields.get("Items") if doc.fields else None
        if items_field is None:
            return CheckpointResult(
                status="fail",
                error_reason="doc.fields['Items'] is missing.",
            )
        items_array = getattr(items_field, "value_array", None) or []
        if len(items_array) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Items.value_array has {len(items_array)} entries, expected at least 3. "
                    f"The sample invoice has 4 to 6 line items per page; if you see fewer "
                    f"the analyzer call may have been truncated."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the checkpoint says `ANALYSIS_RESULT is None`, you never called `.result()` on the poller. `begin_analyze_document` returns an LRO immediately; the actual analysis happens server-side and the client blocks on `.result()` for 20 to 40 seconds on a 5-page PDF.

</details>

<details><summary>Hint 2 (direction)</summary>

Two calls. `poller = doc_intel_client.begin_analyze_document(model_id="prebuilt-invoice", body=pdf_bytes)` then `ANALYSIS_RESULT = poller.result()`. The `pdf_bytes` variable in the cell above is already populated; the `doc_intel_client` is already built.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2's `YOUR CODE` block:

```python
poller = doc_intel_client.begin_analyze_document(
    model_id="prebuilt-invoice", body=pdf_bytes,
)
ANALYSIS_RESULT = poller.result()
```

</details>

**Wallet check.** A 5-page analyze call is 5 pages * ($1.50 / 1,000 pages) = $0.0075. One debug retry doubles that to $0.015. Even with three debugging passes, the spend is under five cents. A $5 coffee buys you about 333 clean analyze calls.

## Task 3: Sum the line-item amounts and confirm they match the invoice total within $0.01

The sum-of-line-items vs invoice-total check is the canonical post-extraction data-quality control. It catches duplicate line items, missed line items, and per-row currency-type confusion. The exam tests this directly: any production invoice ingestion pipeline ships with this check or it does not ship.

Build it in this order:

1. Walk `doc.fields["Items"].value_array`. Each entry is an object with `value_object` mapping keys (`Description`, `Quantity`, `UnitPrice`, `Amount`) to their typed fields.
2. For each item, the per-row amount path is `item.value_object["Amount"].value_currency.amount` (five levels of nesting). Read it as a float.
3. Sum into `LINE_ITEMS_SUM`. Build `LINE_ITEMS` as a list of `{"description", "quantity", "unit_price", "amount"}` dicts so Task 4 can feed it to GPT-4o-mini.
4. Compare `LINE_ITEMS_SUM` to `INVOICE_TOTAL`. The absolute difference must be within $0.01 to absorb floating-point rounding and any genuine penny-rounding in the source document.

**The currency vs float trap.** `item.value_object["Amount"]` is a Currency-typed field, not a float. Trying to `sum()` Currency objects directly throws a TypeError. The float is one access deeper: `.value_currency.amount`.

In [ ]:
# NBVAL_SKIP
# Task 3: Sum line-item amounts, build LINE_ITEMS for Task 4, compare to total.

LINE_ITEMS = []
LINE_ITEMS_SUM = 0.0

items_array = doc.fields["Items"].value_array

for idx, item in enumerate(items_array, start=1):
    obj = item.value_object or {}

    desc_field = obj.get("Description")
    description = desc_field.value_string if desc_field and getattr(desc_field, "value_string", None) else (
        desc_field.content if desc_field else None
    )

    qty_field = obj.get("Quantity")
    quantity = qty_field.value_number if qty_field and getattr(qty_field, "value_number", None) is not None else None

    unit_field = obj.get("UnitPrice")
    unit_price = (
        unit_field.value_currency.amount
        if unit_field and getattr(unit_field, "value_currency", None)
        else None
    )

    # YOUR CODE: Pull the per-row Amount as a float. The path is
    # item.value_object['Amount'].value_currency.amount. Guard for the case
    # where the Amount field is missing on a sparse row.
    # amount_field = obj.get('Amount')
    # amount = amount_field.value_currency.amount if amount_field and getattr(amount_field, 'value_currency', None) else 0.0

    LINE_ITEMS_SUM += amount
    LINE_ITEMS.append({
        "description": description,
        "quantity": quantity,
        "unit_price": unit_price,
        "amount": amount,
    })

delta = abs(LINE_ITEMS_SUM - INVOICE_TOTAL) if INVOICE_TOTAL is not None else None
print(f"Line items: {len(LINE_ITEMS)}")
print(f"Sum of line-item amounts: {LINE_ITEMS_SUM:.2f}")
print(f"InvoiceTotal: {INVOICE_TOTAL:.2f}")
print(f"Absolute difference: {delta:.4f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: LINE_ITEMS_SUM is positive, INVOICE_TOTAL is positive, the
# absolute difference is within $0.01.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if LINE_ITEMS_SUM is None or not isinstance(LINE_ITEMS_SUM, (int, float)) or LINE_ITEMS_SUM <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"LINE_ITEMS_SUM is {LINE_ITEMS_SUM!r}, expected a positive float. "
                    f"Walk the Items.value_array and sum item.value_object['Amount'].value_currency.amount."
                ),
            )

        if INVOICE_TOTAL is None or not isinstance(INVOICE_TOTAL, (int, float)) or INVOICE_TOTAL <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"INVOICE_TOTAL is {INVOICE_TOTAL!r}, expected a positive float "
                    f"captured in Task 2 from doc.fields['InvoiceTotal'].value_currency.amount."
                ),
            )

        delta = abs(LINE_ITEMS_SUM - INVOICE_TOTAL)
        if delta > 0.01:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Line-item sum {LINE_ITEMS_SUM:.2f} does not match invoice total "
                    f"{INVOICE_TOTAL:.2f}; absolute difference {delta:.4f} exceeds $0.01. "
                    f"Check that you read .value_currency.amount and not the raw Currency object."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The Amount field is nested five levels deep: `result.documents[0].fields["Items"].value_array[i].value_object["Amount"].value_currency.amount`. Students who stop at `.value_object["Amount"]` get a Currency object, not a float. Students who try `.value` on that object get `None`.

</details>

<details><summary>Hint 2 (direction)</summary>

Inside the loop: `amount = item.value_object["Amount"].value_currency.amount`. Guard the case where the row's Amount field is missing entirely (sparse row) by falling back to `0.0`. The loop already accumulates into `LINE_ITEMS_SUM` and appends to `LINE_ITEMS`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3's `YOUR CODE` block:

```python
amount_field = obj.get("Amount")
amount = (
    amount_field.value_currency.amount
    if amount_field and getattr(amount_field, "value_currency", None)
    else 0.0
)
```

</details>

**Wallet check.** $0.00 added. Summing a Python list is local computation; no Azure calls. Coffee mocks us by remaining vastly more expensive than the entire session.

## Task 4: Feed the structured JSON to GPT-4o-mini for a one-sentence summary

The deliverable for the ops lead is a single Slack-pastable line. GPT-4o-mini takes the extracted JSON and writes it as `<vendor> invoice for <total> across <N> line items.` Exactly one period, no internal newlines, mentions vendor and total and count.

Build it in this order:

1. Construct an `AzureOpenAI` client backed by `DefaultAzureCredential` via `get_bearer_token_provider`. Target the attached AOAI account's endpoint.
2. Build the user message as the structured JSON of `{vendor, total, line_item_count, line_items}`.
3. Submit a chat completion against the GPT-4o-mini deployment. System message: "Summarize the invoice in exactly one sentence in the form `<vendor> invoice for <total> across <N> line items.`. Do not include newlines."
4. Capture the response into `GPT_SUMMARY` from `response.choices[0].message.content`.

**One-sentence enforcement.** The validator counts periods (must be exactly 1) and rejects internal `\n`. If GPT-4o-mini hedges with a second sentence, tighten the system message; the model is well-behaved on this format with a clear instruction.

In [ ]:
# NBVAL_SKIP
# Task 4: Construct the AzureOpenAI client, prompt GPT-4o-mini for the summary.

token_provider = get_bearer_token_provider(
    AZURE_CREDENTIAL, "https://cognitiveservices.azure.com/.default",
)
aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ACCOUNT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview",
)

structured_payload = {
    "vendor": VENDOR_NAME,
    "total": round(INVOICE_TOTAL, 2),
    "line_item_count": len(LINE_ITEMS),
    "line_items": LINE_ITEMS,
}

system_message = (
    "Summarize the invoice in exactly one sentence in the form "
    "'<vendor> invoice for <total> across <N> line items.'. "
    "Use exactly one period. Do not include newlines. "
    "Format <total> with two decimal places."
)
user_message = json.dumps(structured_payload, default=str)

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

print("Asking GPT-4o-mini to summarize the JSON in one sentence...")
# YOUR CODE: Submit the chat completion against the GPT-4o-mini deployment.
# response = aoai_client.chat.completions.create(
#     model=DEPLOYMENT_NAME, messages=messages, temperature=0,
# )

GPT_SUMMARY = response.choices[0].message.content
print()
print(f"Summary: {GPT_SUMMARY}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: GPT_SUMMARY is non-empty, exactly one period, no newlines,
# contains vendor (case-insensitive), the total formatted to 2 decimals, and
# the line-item count as a digit string.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not GPT_SUMMARY or not isinstance(GPT_SUMMARY, str):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "GPT_SUMMARY is empty or not a string. Capture "
                    "response.choices[0].message.content into GPT_SUMMARY."
                ),
            )

        text = GPT_SUMMARY.strip()
        if not text:
            return CheckpointResult(
                status="fail",
                error_reason="GPT_SUMMARY contains only whitespace.",
            )

        if "\n" in text:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "GPT_SUMMARY contains a newline; expected a single sentence with no "
                    "internal newlines. Tighten the system message."
                ),
            )

        if text.count(".") != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GPT_SUMMARY has {text.count('.')} period(s); expected exactly 1. "
                    f"Tighten the system message to forbid multi-sentence summaries."
                ),
            )

        if not VENDOR_NAME or VENDOR_NAME.lower() not in text.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GPT_SUMMARY does not mention the vendor {VENDOR_NAME!r}. "
                    f"Pass the vendor into the user message as part of the structured JSON."
                ),
            )

        total_str = f"{INVOICE_TOTAL:.2f}"
        if total_str not in text:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GPT_SUMMARY does not contain the total {total_str!r}. "
                    f"Pass round(INVOICE_TOTAL, 2) into the JSON and instruct the model "
                    f"to format with two decimal places."
                ),
            )

        count_str = str(len(LINE_ITEMS))
        if count_str not in text:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GPT_SUMMARY does not contain the line-item count {count_str!r}. "
                    f"Pass line_item_count into the JSON."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the period count is off, the model is hedging with a second sentence; tighten the system message. If the vendor or total substring is missing, the model is paraphrasing; instruct it to render the total with two decimals and to use the exact vendor string from the JSON. No internal newlines means a single line of output.

</details>

<details><summary>Hint 2 (direction)</summary>

The `AzureOpenAI` client is already built with `azure_ad_token_provider` from `DefaultAzureCredential`. One call: `aoai_client.chat.completions.create(model=DEPLOYMENT_NAME, messages=messages, temperature=0)`. The `messages` list and `DEPLOYMENT_NAME` are already in scope. Temperature 0 keeps the format stable across re-runs.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4's `YOUR CODE` block:

```python
response = aoai_client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=messages,
    temperature=0,
)
```

</details>

**Wallet check.** Roughly 400 input + 50 output tokens at GPT-4o-mini rates ($0.15 / $0.60 per 1M) is about $0.00009, or a hundredth of a cent. Session running total is dominated by the Document Intelligence analyze call. Total session damage so far: about one cent. Coffee remains the obvious choice.

## Cleanup

Time to tear it all down. The cell below runs the manifest in reverse-creation order: Document Intelligence resource, GPT-4o-mini deployment, project, hub, resource group. None of these bill at zero traffic; the resource group delete is the cross-cutting safety net. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about one cent for a clean run, under thirty cents even with heavy debugging.** Document Intelligence is the cheapest invoice-extraction surface Azure offers. A buck fifty per thousand pages, so the five-page sample costs less than a cent. The GPT-4o-mini summary at the end is pennies. The trap is debugging the field walk: prebuilt-invoice returns a deeply nested object shape with currency types inside arrays inside fields, and you may iterate the analyzer call three times before you find the right access path. Even then the lab tops out under thirty cents. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. The Checkpoint 3 sanity check (sum of line items equals invoice total) catches a class of extraction errors but not all of them. Walk through three other data-quality checks you would add for production invoice ingestion (think missing fields, currency mismatches, duplicate line items, taxes vs subtotals) and how you would surface each as a metric in Application Insights.

2. Your team is comparing prebuilt-invoice vs a custom-trained invoice model. The prebuilt model handles 80% of vendor formats. The other 20% are vendor-specific layouts (for example, one international vendor uses RTL text and quirky field labels). Walk through what signal would push you to train a custom model, how you would gather training data, and what the operational cost looks like over 12 months relative to the prebuilt-only approach.

## Exam-Style Practice

**Question 1 (Domain 5, prebuilt vs custom vs OCR-then-LLM):**

A team needs to extract structured fields from 50 vendor invoices per day. 80% are standard-layout invoices (US-style with clear "Total" labels). 20% are non-standard (international vendors, custom layouts). The team wants high accuracy on the 80% with the lowest operational cost. Which extraction approach fits?

A. Send every invoice through Document Intelligence prebuilt-invoice; ignore extraction errors on the 20% and let ops re-key those by hand.

B. Send every invoice through Document Intelligence prebuilt-invoice. For invoices where field-level confidence is below 0.7 on critical fields (vendor, total), fall back to a GPT-4o multimodal extraction prompt against the same PDF page images.

C. Train a single custom model on a mixed dataset of all 50 daily invoices and use it for all extraction.

D. Use Document Intelligence prebuilt-layout (OCR only) and a GPT-4o-mini prompt to extract fields from the raw text.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: ignoring 20% of invoices is operationally lazy and the question explicitly says "high accuracy on the 80% with the lowest operational cost," which implies handling the 20% somehow.
- B is correct: this is the Microsoft-documented hybrid extraction pattern. Prebuilt handles the standard layouts cheaply and accurately. The confidence-thresholded fallback to GPT-4o multimodal handles the long-tail layouts at higher per-call cost but only when needed. The two-tier dispatch is the optimal cost/accuracy curve.
- C is wrong: training a custom model on mixed data dilutes accuracy on the dominant 80% pattern. Custom models work best on tight layout families.
- D is wrong: OCR-then-LLM gives up the structured-field model's accuracy advantage and increases token cost. The prebuilt model encodes invoice-field semantics that GPT-4o-mini lacks out of the box.

</details>

**Question 2 (Domain 5, data-quality checks):**

A team extracts vendor invoices via Document Intelligence prebuilt-invoice. Ops noticed last month that some invoices were ingested with subtle errors: a line item was extracted twice, a tax line was conflated with a subtotal, or a currency code was missed. The team wants a programmatic check that runs after every extraction to catch these. Which approach is the most direct data-quality control?

A. Run GPT-4o-mini on the extracted JSON and ask "is this invoice correct?"; reject if the model says no.

B. Compute the sum of line-item amounts and compare to the invoice total within a small tolerance; flag mismatches for human review.

C. Send every extraction to a human reviewer regardless.

D. Disable the prebuilt-invoice model and switch to manual data entry until a custom model is trained.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: LLM-based "is this correct" checks are unreliable for arithmetic. The model can plausibly say yes to an incorrect invoice.
- B is correct: a structural arithmetic check is cheap, deterministic, and catches a meaningful class of errors. This is the data-quality control Checkpoint 3 in this lab implements. Combined with currency code verification, missing-field checks, and duplicate-line-item detection, this is the AI-103 documented post-extraction validation pattern.
- C is wrong: defeats the cost savings of automation.
- D is wrong: prebuilt-invoice's accuracy is well above manual entry's accuracy in benchmarks; switching off is the wrong direction.

</details>